## Setup

In [ ]:
import pandas as pd
import numpy as np
import os
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
import tensorflow as tf
from src.functions import scatter_plot_fn

## Load data

In [ ]:
crbn_binding_data = pd.read_excel('data/mmc3.xlsx', skiprows=1)
crbn_binding_data.head()

## Transform smiles to fingerprints (Morgan, radius 2, n_bits 2048)

In [ ]:
morgan_gen = AllChem.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_fp_list(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * 2048
    fp = morgan_gen.GetFingerprint(mol)
    return(list(fp))

ai_data = pd.DataFrame(crbn_binding_data["Smiles"].apply(smiles_to_fp_list).tolist())
ai_data = pd.concat([ai_data, crbn_binding_data[["Catalog ID", "FP Response"]].rename(columns={"Catalog ID": "id", "FP Response": "binding"}) ], axis=1)
ai_data.head()

## Split data (80% train test split)

In [ ]:
X = ai_data.iloc[:, :-2]
y = ai_data["binding"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Deep learning

#### training

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape = (2048, )),
    tf.keras.layers.Dense(units = 64, activation = 'relu'),
    tf.keras.layers.Dense(units = 64, activation = 'relu'),
    tf.keras.layers.Dense(units = 64, activation = 'relu'),
    tf.keras.layers.Dense(units=1)
])

In [ ]:
model.compile(loss = "mean_squared_error", optimizer=tf.keras.optimizers.Adam(
         learning_rate=0.0001,
        clipnorm=1.0
))

In [ ]:
model.fit(X_train, y_train, epochs=100)

#### Prediction

In [ ]:
y_pred = model.predict(X_test).flatten()

#### Visualization

In [ ]:
scatter_plot_fn(y_test, y_pred, "Neural Network")